# Building a Hotel Recommendation System with Machine Learning
Trip planning almost always starts with choosing a place to stay. With countless booking platforms and options, finding the right hotel can feel overwhelming. In this project, we’ll build a hotel recommendation system in Python that helps surface the most relevant hotel choices for each user using machine learning.

## What this system does
The goal of a hotel recommender is to estimate which properties a user is most likely to book from the full catalog. We’ll leverage signals such as user ratings and review text to learn preferences and produce personalized rankings.

Consider a traveler on a work trip: the system should prioritize hotels that past business travelers have rated highly for location, Wi‑Fi reliability, and check‑in efficiency. By modeling patterns in reviews and ratings, we can match users with hotels that fit their intent.

In the following sections, we’ll walk through the end‑to‑end workflow for building this system in Python—from data preparation and feature engineering to model training and evaluation. We'll use the dataset from [Kaggle](https://www.kaggle.com/datasets/jiashenliu/515k-hotel-reviews-data-in-europe).

## 1. Importing Libraries
First, let’s import the necessary libraries for data manipulation, visualization, and NLP.

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

import warnings

### Configuration
Let's configure the necessary settings.
- Ignore warnings for cleaner output.
- Set default figure size to 12x8 inches for plots.
- Use seaborn's 'darkgrid' style for improved plot aesthetics.

In [29]:
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('darkgrid')

## 2. Data Loading
Next, we will load the dataset into a pandas DataFrame and take a quick look at its structure.

In [30]:
df = pd.read_csv('Hotel_Reviews.csv')

In [31]:
df.head()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.360576,4.915968
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,1403,Great location in nice surroundings the bar a...,26,1,3.8,"[' Leisure trip ', ' Solo traveler ', ' Duplex...",3 days,52.360576,4.915968
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/24/2017,7.7,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,1403,Amazing location and building Romantic setting,8,3,6.7,"[' Leisure trip ', ' Couple ', ' Suite ', ' St...",10 days,52.360576,4.915968


In [32]:
df.sample(5)

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
396356,Parkring 12 01 Innere Stadt 1010 Vienna Austria,251,8/18/2015,9.0,Hotel Am Parkring,United States of America,That we had to leave,7,2304,This is the best European hotel that we have ...,43,3,10.0,"[' Leisure trip ', ' Couple ', ' Double Room '...",716 day,48.205633,16.377862
255356,84 King Street Hammersmith and Fulham London W...,390,6/20/2017,7.8,Best Western Plus Seraphine Hammersmith Hotel,Ireland,Not really a dislike but I had no windows onl...,27,1717,Very comfortable stay Clean room with excelle...,36,11,9.6,"[' Leisure trip ', ' Solo traveler ', ' Standa...",44 days,51.492627,-0.228860
234421,7 9 High Street Kensington Kensington and Chel...,98,11/8/2016,7.4,Seraphine Kensington Gardens Hotel,United Kingdom,Croisants at breakfast were not as fresh as t...,21,597,Staffincluding manager were all very helpful,7,2,10.0,"[' Couple ', ' Executive Double Room ', ' Stay...",268 day,51.502103,-0.187901
50312,140 Gloucester Road Kensington and Chelsea Lon...,528,8/9/2016,8.7,The Bailey s Hotel London,United Kingdom,No views,4,2485,Very fancy and right next to a tube station V...,21,3,9.6,"[' Leisure trip ', ' Couple ', ' Classic Doubl...",359 day,51.493873,-0.182496
505518,Westminster Bridge Road Lambeth London SE1 7UT...,2623,1/5/2016,8.7,Park Plaza Westminster Bridge London,Switzerland,Nothing,3,12158,Everything was great amazing breakfast fantas...,26,56,10.0,"[' Leisure trip ', ' Solo traveler ', ' Superi...",576 day,51.500961,-0.116591


In [33]:
df.shape

(515738, 17)

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515738 entries, 0 to 515737
Data columns (total 17 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Hotel_Address                               515738 non-null  object 
 1   Additional_Number_of_Scoring                515738 non-null  int64  
 2   Review_Date                                 515738 non-null  object 
 3   Average_Score                               515738 non-null  float64
 4   Hotel_Name                                  515738 non-null  object 
 5   Reviewer_Nationality                        515738 non-null  object 
 6   Negative_Review                             515738 non-null  object 
 7   Review_Total_Negative_Word_Counts           515738 non-null  int64  
 8   Total_Number_of_Reviews                     515738 non-null  int64  
 9   Positive_Review                             515738 non-null  object 
 

In [35]:
df.isnull().sum()

Hotel_Address                                    0
Additional_Number_of_Scoring                     0
Review_Date                                      0
Average_Score                                    0
Hotel_Name                                       0
Reviewer_Nationality                             0
Negative_Review                                  0
Review_Total_Negative_Word_Counts                0
Total_Number_of_Reviews                          0
Positive_Review                                  0
Review_Total_Positive_Word_Counts                0
Total_Number_of_Reviews_Reviewer_Has_Given       0
Reviewer_Score                                   0
Tags                                             0
days_since_review                                0
lat                                           3268
lng                                           3268
dtype: int64

## 3. Data Preprocessing
Next, we will preprocess the data.

In [36]:
# Replacing 'United Kingdom' with 'UK' in the 'Hotel_Address' column
df['Hotel_Address'] = df.Hotel_Address.str.replace('United Kingdom', 'UK')

# Splitting the 'Hotel_Address' column into 'Country' with later part of the address
df['Country'] = df.Hotel_Address.apply(lambda x: x.split()[-1])
df.Country.unique()

array(['Netherlands', 'UK', 'France', 'Spain', 'Italy', 'Austria'],
      dtype=object)

In [37]:
# Now, I will drop the unnecessary columns that we don't need for the task of creating a hotel recommendation system.
df.drop(columns=[
    'Additional_Number_of_Scoring',
    'Review_Date',
    'Reviewer_Nationality',
    'Negative_Review', 
    'Review_Total_Negative_Word_Counts',
    'Total_Number_of_Reviews', 
    'Positive_Review',
    'Review_Total_Positive_Word_Counts',
    'Total_Number_of_Reviews_Reviewer_Has_Given', 
    'Reviewer_Score',
    'days_since_review', 'lat', 'lng'], 
    inplace=True
)

In [38]:
# Converting the 'Tags' column to a string from a list
df['Tags'] = df.Tags.apply(lambda x:''.join(literal_eval(x)))

In [39]:
df.head()

,Hotel_Address,Average_Score,Hotel_Name,Tags,Country
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Couple Duplex Double Room Sta...,Netherlands
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Couple Duplex Double Room Sta...,Netherlands
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Family with young children Dup...,Netherlands
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Solo traveler Duplex Double Ro...,Netherlands
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,7.7,Hotel Arena,Leisure trip Couple Suite Stayed 2 nights ...,Netherlands


In [40]:
# Let's convert the 'Tags', and 'Country' columns to lowercase
df['Tags'] = df.Tags.str.lower()
df['Country'] = df.Country.str.lower()

## 4. Text Processing
Pre-process text data and create optimized data structures for faster recommendations.

In [61]:
# Initialize text processing components once (outside the function)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Text preprocessing function"""
    if pd.isna(text):
        return ""
    
    # Convert to lowercase and remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    
    # Tokenize and filter stop words
    tokens = [word for word in text.split() if word not in stop_words and len(word) > 2]
    
    # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

# Pre-process all tags once and cache the results
print("Pre-processing hotel tags...")
df['Processed_Tags'] = df['Tags'].apply(preprocess_text)

# Create a TF-IDF vectorizer for efficient similarity calculation
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8
)

# Fit the vectorizer on all processed tags
print("Creating TF-IDF matrix...")
tfidf_matrix = vectorizer.fit_transform(df['Processed_Tags'])

# Create a mapping for quick country filtering
country_indices = {}
for country in df['Country'].unique():
    country_indices[country] = df[df['Country'] == country].index.tolist()
print('Done!')

Pre-processing hotel tags...
Creating TF-IDF matrix...
Done!


## 5. Recommendation System
Next, we will build a recommendation system.

In [72]:
def recommend_hotel(location, description, top_n=5):
    location = location.lower()
    
    # Check if location exists in our data
    if location not in country_indices:
        print(f"Location '{location}' not found in dataset.")
        return pd.DataFrame(columns=['Hotel_Name', 'Average_Score', 'Hotel_Address'])
    
    # Preprocess the user's description
    processed_description = preprocess_text(description)
    
    # Transform the description using the fitted vectorizer
    description_vector = vectorizer.transform([processed_description])
    
    # Get indices for the specified country
    country_idx = country_indices[location]
    
    # Calculate cosine similarity between description and country hotels
    similarities = cosine_similarity(description_vector, tfidf_matrix[country_idx]).flatten()
    
    # Create a temporary dataframe with results
    country_df = df.iloc[country_idx].copy()
    country_df['similarity'] = similarities
    
    # Sort by similarity (descending) and remove duplicates
    country_df = country_df.sort_values('similarity', ascending=False)
    country_df = country_df.drop_duplicates(subset=['Hotel_Name'], keep='first')
    
    # Sort by average score for hotels with similar similarity scores
    country_df = country_df.sort_values(['similarity', 'Average_Score'], ascending=[False, False])

    
    # Return top recommendations
    rec = country_df[['Hotel_Name', 'Average_Score', 'Hotel_Address']].head(top_n)
    rec.sort_values('Average_Score', ascending=False, inplace=True)
    rec.reset_index(drop=True, inplace=True)
    return rec

## 6. Testing
Let's test the recommendation system.

In [73]:
recommend_hotel('UK','I am going on a honeymoon, I need a honeymoon suite room for 3 nights')

,Hotel_Name,Average_Score,Hotel_Address
0,Staybridge Suites London Stratford,9.2,10b Chestnut Plaza Westfield Stratford City Ol...
1,Dorset Square Hotel,9.0,39 40 Dorset Square Hotel Westminster Borough ...
2,Aloft London Excel,8.7,One Eastern Gateway Royal Victoria Dock Newham...
3,Bentley London,8.4,27 33 Harrington Gardens Kensington and Chelse...
4,Rathbone,8.3,30 Rathbone Street West End Westminster Boroug...


In [74]:
recommend_hotel('Spain', 'Family vacation with kids, need pool and activities')

,Hotel_Name,Average_Score,Hotel_Address
0,Catalonia Plaza Catalunya,8.6,Bergara 11 Eixample 08002 Barcelona Spain
1,Hotel Balmes,8.5,Mallorca 216 Eixample 08008 Barcelona Spain
2,Melia Barcelona Sky 4 Sup,8.4,Diagonal Pere IV 272 Sant Mart 08005 Barcelona...
3,Alexandra Barcelona A DoubleTree by Hilton,8.3,Mallorca 251 Eixample 08008 Barcelona Spain
4,Hilton Diagonal Mar Barcelona,7.9,Passeig del Taulat 262 264 Sant Mart 08019 Bar...


## 7. Conclusion
 
This hotel recommendation system successfully leverages TF-IDF vectorization and cosine similarity to provide personalized hotel suggestions based on user preferences and location. The system processes hotel tags through NLP techniques (lemmatization, stopword removal) to create meaningful feature vectors for similarity matching.

**Key Insights:**
- Text preprocessing significantly improves recommendation quality by standardizing hotel descriptions
- Location-based filtering ensures relevant geographical results
- Combining similarity scores with average ratings provides balanced recommendations

**Future Improvements:**
- Implement collaborative filtering to incorporate user behavior patterns
- Add sentiment analysis of reviews for more nuanced recommendations
- Include price range and amenities as additional filtering criteria
- Develop a real-time scoring system that updates based on recent reviews
- Create user profiles to track preferences and improve personalization over time